# Figure 2 panels

Builds every Figure 2 panel from the canonical normalised object. Each panel is saved as vector PDF (scatter layers rasterised) into `07_figures/figure2/`, ready to assemble in Illustrator.

**Panels**

| id | content | source |
|---|---|---|
| 2A | Tyser CS7 embryo UMAP, reference's own `cluster_id` | Tyser |
| 2B | Control gastruloid UMAP by legacy annotation, 42h vs 72h | master |
| 2C | Dot plot: APLNR/APELA x legacy state, controls, split by time | master |
| 2D | Dot plot: APLNR/APELA x embryo tissue | Tyser |
| 2E | MAGIC-imputed APLNR/APELA on gastruloid UMAP | master |
| 2F | MAGIC-imputed APLNR/APELA on embryo UMAP | Tyser |

**Deliberately excluded:** `embryo_cluster_id_pred`. The mapping diagnostics showed 88.9% collapse onto Primitive Streak with no reference neighbours, so per-cell embryo labels are not used for any panel here. Panels 2A/2D use the reference's own annotations, which need no mapping.

Figure 2 is **controls only**. Knockdown comparisons belong in the KD figure.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scipy.sparse as sp
import anndata as ad
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
plt.rcParams["pdf.fonttype"] = 42   # editable text in Illustrator
plt.rcParams["ps.fonttype"] = 42

import sys
sys.path.insert(0, str(Path.cwd().parent / "utils"))   # run notebooks from their own folder
from config import ROOT, PROJECT_DIR, CELLRANGER_DIR

MASTER = ROOT / "00_DATA_FROZEN" / "03_analysis_ready" / "adata_master_normalized.h5ad"
TYSER  = ROOT / "00_DATA_FROZEN" / "03_analysis_ready" / "tyser_embryo_genes_for_fig2.h5ad"
FIG    = ROOT / "07_figures" / "figure2"
TAB    = ROOT / "06_tables"  / "figure2"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

GENES = ["APLNR", "APELA"]

# Legacy state display order. 'stressed' (n=8) is dropped from figures.
LEGACY_ORDER = ["pluripotent", "AE-LC", "DE", "aPM-LC", "LPM-LC",
                "eNasM", "lNasM", "aPS-LC", "pPS-LC", "PGC-LC", "CS-LC"]
DROP_STATES = ["stressed"]

GRAY_RED = LinearSegmentedColormap.from_list("gray_red", ["lightgray", "red"])

for n, p in [("master", MASTER), ("tyser", TYSER)]:
    print(f"{n:8s} {'OK ' if p.exists() else 'MISSING'} {p.name}")
if not MASTER.exists():
    raise FileNotFoundError("Run 07_canonical_master_build.ipynb first.")

## Load and subset

In [ ]:
adata = ad.read_h5ad(MASTER)
print("master:", adata.shape)

# guard: this notebook assumes normalised data
if "normalization" not in adata.uns:
    raise RuntimeError("Object is not the normalised master. Run 07_canonical_master_build.ipynb.")
print("normalisation:", adata.uns["normalization"]["method"])

def sym_index(a, gene):
    """Return the var index name for a plain gene symbol."""
    hit = a.var.index[a.var["symbol"] == gene] if "symbol" in a.var else a.var.index[a.var_names == gene]
    if len(hit) == 0:
        raise KeyError(f"{gene} not found")
    return hit[0]

GENE_KEY = {g: sym_index(adata, g) for g in GENES}
print("gene keys:", GENE_KEY)

# Controls only, drop tiny junk states, set display order
ctrl = adata[(adata.obs["condition"].astype(str) == "Ctrl")
             & (~adata.obs["predicted_leiden_sub_joint"].astype(str).isin(DROP_STATES))].copy()

present = [s for s in LEGACY_ORDER if s in set(ctrl.obs["predicted_leiden_sub_joint"].astype(str))]
unlisted = sorted(set(ctrl.obs["predicted_leiden_sub_joint"].astype(str)) - set(LEGACY_ORDER))
if unlisted:
    print("WARNING: states present but not in LEGACY_ORDER:", unlisted)

ctrl.obs["legacy_state"] = pd.Categorical(
    ctrl.obs["predicted_leiden_sub_joint"].astype(str), categories=present, ordered=True)

print(f"\ncontrol cells: {ctrl.n_obs:,}")
print(ctrl.obs.groupby(["site", "timepoint_true"], observed=True).size().to_string())

## Panel 2A — Tyser CS7 embryo reference

The reference's own annotations on its own UMAP. No label transfer involved.

In [ ]:
tyser = ad.read_h5ad(TYSER)
print("tyser:", tyser.shape)

TY_GENE_KEY = {g: sym_index(tyser, g) for g in GENES}
print("embryo gene keys:", TY_GENE_KEY)

fig, ax = plt.subplots(figsize=(5.5, 5))
sc.pl.embedding(tyser, basis="X_umap", color="cluster_id", ax=ax, show=False,
                frameon=False, size=28, legend_loc="right margin",
                title="Human CS7 embryo (Tyser et al.)")
fig.savefig(FIG / "2A_embryo_umap.pdf", bbox_inches="tight")
plt.show()

## Panel 2B — Control gastruloids across time

Same UMAP coordinates for both timepoints so the panels are directly comparable. Cells from the other timepoint are drawn in light grey as context.

In [ ]:
UMAP_BASIS = "X_umap_joint"
if UMAP_BASIS not in ctrl.obsm:
    raise KeyError(f"{UMAP_BASIS} missing. obsm: {list(ctrl.obsm.keys())}")

xy_all = np.asarray(ctrl.obsm[UMAP_BASIS])
times = ["42h", "72h"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, t in zip(axes, times):
    mask = (ctrl.obs["timepoint_true"].astype(str) == t).to_numpy()
    ax.scatter(xy_all[~mask, 0], xy_all[~mask, 1], s=3, c="#e8e8e8",
               linewidths=0, rasterized=True, zorder=1)
    sub = ctrl[mask]
    xy = np.asarray(sub.obsm[UMAP_BASIS])
    for state in sub.obs["legacy_state"].cat.categories:
        m = (sub.obs["legacy_state"].astype(str) == state).to_numpy()
        if m.sum() == 0:
            continue
        ax.scatter(xy[m, 0], xy[m, 1], s=4, linewidths=0, rasterized=True,
                   zorder=2, label=f"{state} ({m.sum():,})")
    ax.set_title(f"Control {t}  (n={mask.sum():,})")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)
    ax.set_aspect("equal")

axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
                frameon=False, markerscale=3, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "2B_control_umap_by_time.pdf", bbox_inches="tight")
plt.show()

## Panel 2C — APLNR / APELA across gastruloid tissues

Dot size is the fraction of cells expressing; colour is mean normalised expression among all cells in the group. Colour scales are **not** shared with the embryo panel — see the note under 2D.

In [ ]:
for t in times:
    sub = ctrl[ctrl.obs["timepoint_true"].astype(str) == t].copy()
    sub.obs["legacy_state"] = sub.obs["legacy_state"].cat.remove_unused_categories()
    dp = sc.pl.dotplot(
        sub, {g: [GENE_KEY[g]] for g in GENES}, groupby="legacy_state",
        use_raw=False, swap_axes=True, standard_scale=None,
        dot_max=0.6, dot_min=0.0, smallest_dot=12, figsize=(7, 2.4),
        title=f"Control {t}", return_fig=True,
    )
    dp.savefig(FIG / f"2C_dotplot_control_{t}.pdf", bbox_inches="tight")
    dp.show()

# plot-ready table behind the panel
rows = []
for t in times:
    sub = ctrl[ctrl.obs["timepoint_true"].astype(str) == t]
    for state in sub.obs["legacy_state"].cat.categories:
        cells = sub[sub.obs["legacy_state"].astype(str) == state]
        if cells.n_obs == 0:
            continue
        for g in GENES:
            v = cells[:, GENE_KEY[g]].X
            v = np.asarray(v.todense()).ravel() if sp.issparse(v) else np.asarray(v).ravel()
            rows.append({"timepoint": t, "state": state, "gene": g, "n_cells": cells.n_obs,
                         "pct_expressing": round(float((v > 0).mean() * 100), 2),
                         "mean_expression": round(float(v.mean()), 4)})
dot_tbl = pd.DataFrame(rows)
dot_tbl.to_csv(TAB / "2C_gastruloid_dotplot_values.csv", index=False)
display(dot_tbl.head(12))

## Panel 2D — APLNR / APELA across embryo tissues

**Scale caveat.** The Tyser `.raw` is log-normalised but its `target_sum` is unknown, while the master uses 1e4. Panels 2C and 2D therefore use independent colour bars, and no quantitative claim should be made comparing expression magnitude or % cells expressing between the two datasets — that is confounded by gene panel size (21,080 vs 38,606) and sequencing depth. The valid comparison is qualitative: *which tissues* carry the signal.

In [ ]:
dp = sc.pl.dotplot(
    tyser, {g: [TY_GENE_KEY[g]] for g in GENES}, groupby="cluster_id",
    use_raw=False, swap_axes=True, standard_scale=None,
    dot_max=0.6, dot_min=0.0, smallest_dot=12, figsize=(7, 2.4),
    title="Human CS7 embryo", return_fig=True,
)
dp.savefig(FIG / "2D_dotplot_embryo.pdf", bbox_inches="tight")
dp.show()

rows = []
for ct in tyser.obs["cluster_id"].astype(str).unique():
    cells = tyser[tyser.obs["cluster_id"].astype(str) == ct]
    for g in GENES:
        v = cells[:, TY_GENE_KEY[g]].X
        v = np.asarray(v.todense()).ravel() if sp.issparse(v) else np.asarray(v).ravel()
        rows.append({"tissue": ct, "gene": g, "n_cells": cells.n_obs,
                     "pct_expressing": round(float((v > 0).mean() * 100), 2),
                     "mean_expression": round(float(v.mean()), 4)})
emb_tbl = pd.DataFrame(rows).sort_values(["gene", "mean_expression"], ascending=[True, False])
emb_tbl.to_csv(TAB / "2D_embryo_dotplot_values.csv", index=False)
display(emb_tbl)

## MAGIC imputation

APLNR is detected in only ~5.9% of cells, so per-cell plots of raw normalised values are sparse. MAGIC diffuses expression over the kNN graph to recover structure.

Two decisions worth stating in your methods:

1. **Run per site.** Michigan and Chicago are separate captures with different depth. Imputing across them would diffuse signal between biological replicates and inflate apparent consistency.
2. **Fit on HVGs.** MAGIC on the full 38,606-gene space is computationally infeasible at 45k cells. The graph is built from highly variable genes; imputed values are returned for the genes of interest.

Imputed values land in `obs['APLNR_magic']` / `obs['APELA_magic']` rather than a layer, so the shape stays consistent with the full object.

In [ ]:
try:
    import magic
except ImportError as e:
    raise ImportError(
        "magic-impute not installed. In your scvi env:\n"
        "    pip install magic-impute\n"
        "(imports as `magic`; do NOT install the unrelated PyPI package `magic`)"
    ) from e

N_HVG = 2000
MAGIC_KNN = 5
MAGIC_T = 1

def run_magic_per_site(a, genes, gene_key, n_hvg=N_HVG, knn=MAGIC_KNN, t=MAGIC_T):
    """MAGIC fit separately within each site. Returns DataFrame indexed like a.obs."""
    out = pd.DataFrame(index=a.obs_names, columns=[f"{g}_magic" for g in genes], dtype=float)

    for site in a.obs["site"].astype(str).unique():
        m = (a.obs["site"].astype(str) == site).to_numpy()
        sub = a[m].copy()
        print(f"  {site}: {sub.n_obs:,} cells")

        hv = sub.copy()
        sc.pp.highly_variable_genes(hv, n_top_genes=n_hvg, flavor="seurat")
        keep = set(hv.var_names[hv.var["highly_variable"]]) | {gene_key[g] for g in genes}
        sub = sub[:, sorted(keep)].copy()

        X = sub.X.toarray() if sp.issparse(sub.X) else np.asarray(sub.X)
        df = pd.DataFrame(X, index=sub.obs_names, columns=sub.var_names)

        op = magic.MAGIC(knn=knn, t=t, random_state=0, verbose=0)
        imputed = op.fit_transform(df, genes=[gene_key[g] for g in genes])

        for g in genes:
            out.loc[sub.obs_names, f"{g}_magic"] = imputed[gene_key[g]].values

    return out

print("Gastruloid (controls + KD, so the KD figure can reuse this):")
magic_all = run_magic_per_site(adata, GENES, GENE_KEY)
for g in GENES:
    adata.obs[f"{g}_magic"] = magic_all[f"{g}_magic"]
ctrl.obs[[f"{g}_magic" for g in GENES]] = magic_all.loc[ctrl.obs_names, [f"{g}_magic" for g in GENES]]

print(adata.obs[[f"{g}_magic" for g in GENES]].describe().round(4).to_string())

In [ ]:
# Embryo: single capture, so MAGIC is fit once over all cells
print("Embryo:")
hv = tyser.copy()
sc.pp.highly_variable_genes(hv, n_top_genes=N_HVG, flavor="seurat")
keep = set(hv.var_names[hv.var["highly_variable"]]) | {TY_GENE_KEY[g] for g in GENES}
sub = tyser[:, sorted(keep)].copy()

X = sub.X.toarray() if sp.issparse(sub.X) else np.asarray(sub.X)
df = pd.DataFrame(X, index=sub.obs_names, columns=sub.var_names)

op = magic.MAGIC(knn=MAGIC_KNN, t=MAGIC_T, random_state=0, verbose=0)
imputed = op.fit_transform(df, genes=[TY_GENE_KEY[g] for g in GENES])

for g in GENES:
    tyser.obs[f"{g}_magic"] = imputed[TY_GENE_KEY[g]].values

print(tyser.obs[[f"{g}_magic" for g in GENES]].describe().round(4).to_string())

## Panel 2E — imputed APLNR / APELA on the gastruloid UMAP

Colour limits are shared across timepoints per gene, so 42h and 72h are directly comparable. They are **not** shared with panel 2F.

In [ ]:
lims = {g: (0.0, float(np.nanpercentile(ctrl.obs[f"{g}_magic"], 99))) for g in GENES}
print("colour limits:", {g: (round(a, 3), round(b, 3)) for g, (a, b) in lims.items()})

fig, axes = plt.subplots(len(GENES), 2, figsize=(11, 4.6 * len(GENES)))
axes = np.atleast_2d(axes)

for r, g in enumerate(GENES):
    vmin, vmax = lims[g]
    for c, t in enumerate(times):
        ax = axes[r, c]
        mask = (ctrl.obs["timepoint_true"].astype(str) == t).to_numpy()
        ax.scatter(xy_all[~mask, 0], xy_all[~mask, 1], s=3, c="#f0f0f0",
                   linewidths=0, rasterized=True, zorder=1)
        vals = ctrl.obs[f"{g}_magic"].to_numpy()[mask]
        o = np.argsort(vals)  # high-expressing cells drawn on top
        sctr = ax.scatter(xy_all[mask][o, 0], xy_all[mask][o, 1], s=4, c=vals[o],
                          cmap=GRAY_RED, vmin=vmin, vmax=vmax,
                          linewidths=0, rasterized=True, zorder=2)
        ax.set_title(f"{g} — control {t}")
        ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False); ax.set_aspect("equal")
    fig.colorbar(sctr, ax=axes[r, :].tolist(), fraction=0.025, label=f"{g} (MAGIC)")

fig.savefig(FIG / "2E_gastruloid_magic_umap.pdf", bbox_inches="tight")
plt.show()

## Panel 2F — imputed APLNR / APELA on the embryo UMAP

In [ ]:
ty_xy = np.asarray(tyser.obsm["X_umap"])

fig, axes = plt.subplots(1, len(GENES), figsize=(5.5 * len(GENES), 5))
axes = np.atleast_1d(axes)

for ax, g in zip(axes, GENES):
    vals = tyser.obs[f"{g}_magic"].to_numpy()
    vmax = float(np.nanpercentile(vals, 99))
    o = np.argsort(vals)
    sctr = ax.scatter(ty_xy[o, 0], ty_xy[o, 1], s=26, c=vals[o], cmap=GRAY_RED,
                      vmin=0, vmax=vmax, linewidths=0, rasterized=True)
    ax.set_title(f"{g} — CS7 embryo")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False); ax.set_aspect("equal")
    fig.colorbar(sctr, ax=ax, fraction=0.046, label=f"{g} (MAGIC)")

fig.tight_layout()
fig.savefig(FIG / "2F_embryo_magic_umap.pdf", bbox_inches="tight")
plt.show()

## Save objects carrying the imputed values

Written so the KD figure notebook can reuse the same MAGIC values rather than recomputing them (and risking a different graph).

In [ ]:
SAVE = True

MASTER_MAGIC = ROOT / "04_processed" / "adata_master_normalized_magic.h5ad"
TYSER_MAGIC  = ROOT / "04_processed" / "tyser_embryo_genes_for_fig2_magic.h5ad"

if SAVE:
    ad.settings.allow_write_nullable_strings = True
    adata.write_h5ad(MASTER_MAGIC)
    tyser.write_h5ad(TYSER_MAGIC)
    print(f"wrote {MASTER_MAGIC.name} ({MASTER_MAGIC.stat().st_size/1e9:.2f} GB)")
    print(f"wrote {TYSER_MAGIC.name} ({TYSER_MAGIC.stat().st_size/1e6:.0f} MB)")
else:
    print("SAVE = False; nothing written.")

print("\nPanels written to", FIG)
for p in sorted(FIG.glob("*.pdf")):
    print("   ", p.name)

## Notes for the methods section

- Expression values are library-size normalised to 1e4 and log1p-transformed. Raw counts retained in `layers['counts']`.
- MAGIC (`knn=5`, `t=1`) fit independently within each site (Michigan, Chicago) on the top 2,000 highly variable genes plus the genes of interest. Imputation is used for visualisation only — the dot plots in 2C/2D show unimputed values.
- APLNR is detected in 5.9% of gastruloid cells and APELA in 17.4%; imputation is doing substantial work for APLNR and this should be stated.
- `n = 2` biological replicates (independent differentiations at two sites). Site is confounded with processing batch.
- Timepoints are 42h and 72h. Earlier files label 42h as `48h`; `timepoint_original` preserves this.
- Embryo tissue identities are the reference's own annotations (Tyser et al.), not transferred labels.

In [ ]:
c = adata.obs["prediction_confidence_joint"]
print(c.describe())
print(adata.obs.groupby("predicted_leiden_sub_joint", observed=True)["prediction_confidence_joint"]
      .agg(["count", "median"]).sort_values("median"))